# Lab 3: Hugging Face versus an inference engine

You run the same model in two ways and measure both:

1. **Hugging Face `transformers`**: the model lives in this notebook and
   answers with `model.generate()`;
2. **vLLM**: an inference engine that runs the model as a server; the notebook
   sends it HTTP requests.

For each, you write and run a **speed benchmark** (time to first token, time
per output token, tokens per second) and an **evaluation** (accuracy on your
Lab 2 test set with the Lab 2 prompt and scorer). At the end you design one
experiment of your own.

All the code is in the notebook: read each cell before you run it. The same
steps as terminal commands, using ready-made scripts, are in `README.md`.

**Where to run it:** JupyterLab **with a GPU** on the GPU side of Roihu, with
the `python-pytorch/2.13` module and the packages you installed in Lab 2
(Lab 2, step 3). vLLM is started from the notebook with its own module.

## 0. Settings and the Lab 2 task

In [ ]:
import json
import os
import time
from pathlib import Path

LAB3 = Path.home() / "LLM-Course-2026" / "Week-3" / "Lab-3"
LAB2 = Path.home() / "LLM-Course-2026" / "Week-2" / "Lab-2"
TASK_CONFIG = LAB2 / "configs" / "my-task.json"   # your Lab 2 config; or "roihu.json" for the example
MODEL = "Qwen/Qwen3.5-0.8B"
os.environ.setdefault("HF_HOME", f"/scratch/project_2021893/{os.environ['USER']}/hf")
os.chdir(LAB3)

We reuse the Lab 2 task exactly: its labels, its test set, and its prompt and
answer extraction from `lab2lib/prompting.py`. That way the accuracy here is
comparable with Lab 2.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("prompting", LAB2 / "lab2lib" / "prompting.py")
prompting = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prompting)

task = json.loads(TASK_CONFIG.read_text())
LABELS = task["task"]["labels"]
test = [json.loads(line) for line in open(LAB2 / task["data"]["test"]) if line.strip()]
test = test[:100]                                   # 100 examples keep this quick

def task_messages(example):
    """The zero-shot Lab 2 prompt as chat messages."""
    return [
        {"role": "system", "content": prompting.TASK_SYSTEM},
        {"role": "user", "content": prompting.task_prompt(example, [])},
    ]

print(len(test), "test examples; labels:", LABELS)
print(task_messages(test[0])[1]["content"])

For the speed tests we use varied, task-independent prompts, and make every
answer exactly the same length, so that every request does the same amount of
work.

In [ ]:
prompts = [line.strip() for line in open(LAB3 / "configs" / "benchmark-prompts.txt")
           if line.strip() and not line.startswith("#")]
OUTPUT_TOKENS = 128

def speed_messages(i):
    # "Request i" makes every prompt unique, so nothing can be reused between requests.
    return [{"role": "user", "content": f"{prompts[i % len(prompts)]}\nRequest {i}."}]

print(len(prompts), "benchmark prompts, e.g.:", prompts[0])

## 1. Hugging Face

### Load the model

The weights are copied into GPU memory. Prompts are padded **on the left**, so
that in a batch every prompt ends right where generation starts.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(device).eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters on {device}")

In [ ]:
def hf_generate(batch_of_messages, max_new_tokens, fixed_length=False):
    """Generate answers for a batch of chats with model.generate()."""
    texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
             for m in batch_of_messages]
    inputs = tokenizer(texts, return_tensors="pt", padding=True).to(device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=False,                                   # greedy
            max_new_tokens=max_new_tokens,
            min_new_tokens=max_new_tokens if fixed_length else None,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return [tokenizer.decode(row, skip_special_tokens=True) for row in new_tokens]

print(hf_generate([task_messages(test[0])], max_new_tokens=32))

### Speed benchmark

`generate()` does not tell us when the first token appeared, so we time two
calls: one that produces only **one** token (that is the time to first token:
reading the prompt), and one that produces the full answer. The rest of the
time, divided by the remaining tokens, is the time per output token.

"Concurrency" for Hugging Face means the **batch size**: how many prompts go
through `generate()` together.

In [ ]:
def sync():
    if device == "cuda":
        torch.cuda.synchronize()        # wait until the GPU has really finished

def hf_benchmark(concurrency, n_requests=16):
    ttft, tpot, start_all = [], [], time.perf_counter()
    for first in range(0, n_requests, concurrency):
        batch = [speed_messages(i) for i in range(first, first + concurrency)]
        sync(); t0 = time.perf_counter()
        hf_generate(batch, 1, fixed_length=True)
        sync(); t1 = time.perf_counter()
        hf_generate(batch, OUTPUT_TOKENS, fixed_length=True)
        sync(); t2 = time.perf_counter()
        ttft += [(t1 - t0) * 1000] * len(batch)
        tpot += [(t2 - t1 - (t1 - t0)) * 1000 / (OUTPUT_TOKENS - 1)] * len(batch)
    elapsed = time.perf_counter() - start_all
    return {"concurrency": concurrency,
            "ttft_ms": sorted(ttft)[len(ttft) // 2],                # median
            "tpot_ms": sorted(tpot)[len(tpot) // 2],
            "tokens_per_s": n_requests * OUTPUT_TOKENS / elapsed}

hf_benchmark(1, n_requests=2)                                     # warm-up, not counted
results = {"hf": [hf_benchmark(c) for c in (1, 8)]}
results["hf"]

### Evaluation

The same test examples, the Lab 2 prompt, greedy decoding, and the Lab 2
answer extraction. We answer 16 examples per `generate()` call.

In [ ]:
def evaluate(answer_fn, batch_size=16):
    start = time.perf_counter()
    answers = []
    for first in range(0, len(test), batch_size):
        answers += answer_fn([task_messages(e) for e in test[first:first + batch_size]])
    seconds = time.perf_counter() - start
    predictions = [prompting.extract_task_label(a, LABELS) for a in answers]
    accuracy = sum(p == e["label"] for p, e in zip(predictions, test)) / len(test)
    return {"accuracy": accuracy, "seconds": seconds, "predictions": predictions, "answers": answers}

accuracy = {"hf": evaluate(lambda batch: hf_generate(batch, max_new_tokens=32))}
print(f"Hugging Face: accuracy {accuracy['hf']['accuracy']:.1%} in {accuracy['hf']['seconds']:.1f} s")

Free the GPU memory before starting vLLM:

In [ ]:
del model
torch.cuda.empty_cache() if device == "cuda" else None

## 2. vLLM

### Start the server

vLLM comes from its own Roihu module. We start it as a background process
and wait until it answers. Its log is written to `vllm.log`.

In [ ]:
import subprocess
import urllib.request

PORT = 8000
VLLM_SERVE = (f"vllm serve {MODEL} --served-model-name task --port {PORT} "
              f"--max-model-len 4096 --gpu-memory-utilization 0.8 --no-enable-prefix-caching")

def start_server(extra_args=""):
    command = f"module purge && module load python-vllm && PYTHONNOUSERSITE=1 {VLLM_SERVE} {extra_args}"
    server = subprocess.Popen(["bash", "-lc", command], stdout=open("vllm.log", "w"),
                              stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(600):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
            print("Server ready")
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError("The server stopped; see vllm.log")
            time.sleep(1)
    raise RuntimeError("The server did not start in 10 minutes; see vllm.log")

def stop_server(server):
    os.killpg(server.pid, 15)                 # stop vLLM and its helper processes
    server.wait()

server = start_server()

### Talk to it

The server speaks the OpenAI chat API: we send JSON over HTTP and get JSON
back. This is all a client needs.

In [ ]:
URL = f"http://127.0.0.1:{PORT}/v1/chat/completions"

def post(payload):
    request = urllib.request.Request(URL, data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"})
    return urllib.request.urlopen(request, timeout=300)

def vllm_answer(messages, max_tokens=32):
    payload = {"model": "task", "messages": messages, "temperature": 0, "max_tokens": max_tokens}
    with post(payload) as response:
        return json.load(response)["choices"][0]["message"]["content"]

print(vllm_answer(task_messages(test[0])))

### Speed benchmark

With `"stream": true` the server sends the answer piece by piece, so we can
see exactly when the first token arrives. `ignore_eos` makes every answer
exactly `OUTPUT_TOKENS` long.

"Concurrency" for vLLM means how many requests are **in flight at the same
time**; we send them from parallel threads.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def vllm_timed(i):
    payload = {"model": "task", "messages": speed_messages(i), "temperature": 0,
               "max_tokens": OUTPUT_TOKENS, "ignore_eos": True, "stream": True}
    start = time.perf_counter()
    first = last = None
    with post(payload) as response:
        for line in response:                                  # one event per line
            line = line.decode().strip()
            if not line.startswith("data:") or line.endswith("[DONE]"):
                continue
            choices = json.loads(line[5:]).get("choices")
            if choices and choices[0]["delta"].get("content"):
                last = time.perf_counter()
                first = first or last
    return {"ttft_ms": (first - start) * 1000,
            "tpot_ms": (last - first) * 1000 / (OUTPUT_TOKENS - 1)}

def vllm_benchmark(concurrency, n_requests=16):
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        rows = list(pool.map(vllm_timed, range(n_requests)))
    elapsed = time.perf_counter() - start
    return {"concurrency": concurrency,
            "ttft_ms": sorted(r["ttft_ms"] for r in rows)[n_requests // 2],
            "tpot_ms": sorted(r["tpot_ms"] for r in rows)[n_requests // 2],
            "tokens_per_s": n_requests * OUTPUT_TOKENS / elapsed}

vllm_benchmark(1, n_requests=3)                                  # warm-up, not counted
results["vllm"] = [vllm_benchmark(c) for c in (1, 8)]
results["vllm"]

### Evaluation

The same `evaluate()` function as for Hugging Face; only the way of answering
changes: 16 requests sent in parallel.

In [ ]:
def vllm_batch(batch):
    with ThreadPoolExecutor(max_workers=len(batch)) as pool:
        return list(pool.map(vllm_answer, batch))

accuracy["vllm"] = evaluate(vllm_batch)
print(f"vLLM: accuracy {accuracy['vllm']['accuracy']:.1%} in {accuracy['vllm']['seconds']:.1f} s")

## 3. Compare

Which is faster for one request, and for 8 at a time? Is the accuracy the
same? Explain what you see with the lecture: static versus continuous
batching, prefill and decode.

In [ ]:
print(f"{'run':<6}{'concurrency':>12}{'first token ms':>16}{'per token ms':>14}{'tokens/s':>10}")
for name, rows in results.items():
    for r in rows:
        print(f"{name:<6}{r['concurrency']:>12}{r['ttft_ms']:>16.0f}{r['tpot_ms']:>14.2f}{r['tokens_per_s']:>10.0f}")
for name, a in accuracy.items():
    print(f"{name}: accuracy {a['accuracy']:.1%}, {a['seconds']:.1f} s for {len(test)} examples")

In [ ]:
# Where do the two give different answers?
for e, a, b, ta, tb in zip(test, accuracy["hf"]["predictions"], accuracy["vllm"]["predictions"],
                           accuracy["hf"]["answers"], accuracy["vllm"]["answers"]):
    if a != b:
        print(f"{e['id']}  gold={e['label']}\n  hf:   {a}  {ta.strip()[:60]}\n  vllm: {b}  {tb.strip()[:60]}")

In [ ]:
import matplotlib.pyplot as plt

for name, rows in results.items():
    plt.plot([r["concurrency"] for r in rows], [r["tokens_per_s"] for r in rows], marker="o", label=name)
plt.xscale("log", base=2)
plt.xlabel("requests at the same time")
plt.ylabel("output tokens per second (all requests)")
plt.legend()
plt.show()

## 4. Your own experiment

Choose **one** question, change the code below, and explain the result with a
table or plot and a paragraph. Use the functions defined above.

- **A. Where is the knee?** Raise the concurrency: when does adding requests
  stop paying off? (example below)
- **B. Prompt length.** Make the prompts much longer (e.g. repeat a paragraph)
  and compare the time to first token. Why does it change?
- **C. Predict, then measure.** Memory bandwidth (about 4 TB/s) divided by the
  model size (about 1.6 GB in bf16) gives an upper limit for one user. How
  close does concurrency 1 get, and why not closer?
- **D. Your own workload.** Prompts and an answer length typical of your task
  (`prompts`, `OUTPUT_TOKENS`). Does vLLM's advantage hold?

In [ ]:
# Example for A: vLLM with more and more requests at the same time.
results["vllm-knee"] = [vllm_benchmark(c, n_requests=max(16, c)) for c in (1, 2, 4, 8, 16, 32, 64)]
results["vllm-knee"]

Run the comparison and plot cells in section 3 again to see your new results.

## 5. Stop the server

A running server keeps its GPU memory. Stop it when you are done, and save
your results.

In [ ]:
stop_server(server)
Path("results").mkdir(exist_ok=True)
Path("results/notebook-results.json").write_text(json.dumps(
    {"speed": results, "accuracy": {k: v["accuracy"] for k, v in accuracy.items()}}, indent=2))
print("Saved results/notebook-results.json")

## Optional: FP8 weights

Restart vLLM with 8-bit weights and measure again with the same functions.

In [ ]:
server = start_server("--quantization fp8")
vllm_benchmark(1, n_requests=3)                                  # warm-up
results["vllm-fp8"] = [vllm_benchmark(c) for c in (1, 8)]
accuracy["vllm-fp8"] = evaluate(vllm_batch)
stop_server(server)
results["vllm-fp8"], accuracy["vllm-fp8"]["accuracy"]

## Optional: JSON with and without a schema

Ask for the answer as `{"label": ...}`, once with the prompt only and once
with a JSON schema that the server enforces while generating.

In [ ]:
server = start_server()
schema = {"type": "object", "properties": {"label": {"type": "string", "enum": LABELS}},
          "required": ["label"], "additionalProperties": False}

def json_answer(example, use_schema):
    messages = [{"role": "user", "content": prompting.task_example(example, include_answer=False)
                 + f'\n\nAnswer with JSON only, like {{"label": "..."}}, using one of: {", ".join(LABELS)}.'}]
    payload = {"model": "task", "messages": messages, "temperature": 0, "max_tokens": 64}
    if use_schema:
        payload["response_format"] = {"type": "json_schema", "json_schema": {"name": "answer", "schema": schema}}
    with post(payload) as response:
        return json.load(response)["choices"][0]["message"]["content"]

for use_schema in (False, True):
    invalid = correct = 0
    for e in test:
        try:
            label = json.loads(json_answer(e, use_schema))["label"]
            correct += label == e["label"]
        except (json.JSONDecodeError, KeyError, TypeError):
            invalid += 1
    print(f"schema={use_schema}: invalid {invalid / len(test):.0%}, accuracy {correct / len(test):.0%}")
stop_server(server)